# LightGBM + SHAP dashboard data-quality companion

## tl;dr

The three headway files reconcile to **217,610 event rows across 34 effective configurations** after retaining one copy of the repeated S1 HDV baseline. The files support the dashboard's microscopic, policy-lever, and combined modes, with a documented reduced microscopic feature set because lane count, speed limit, and intersection-control fields are not present in these CSVs.

## Context & Methods

This notebook is the reproducible source audit for the interactive LightGBM + SHAP result module. The intended grain is one extracted vehicle-conflict event, and every source row already satisfies minTTC ≤ 1.0 s.

### Key Assumptions

- S1 is an identical HDV-only baseline in all three files; the all-headway model retains only its 0.6-file copy.
- A headway-specific model fixes headway by design, so headway is omitted as a predictor.
- Vehicle IDs and post-outcome timing fields are excluded from the feature matrix.
- SHAP values are predictive associations, not causal effects.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd

PROJECT_ROOT = Path(os.environ.get(
    'AV_SAFETY_PROJECT',
    r'C:/Users/am.taheri/Documents/AvicenAI/AV_Safety_Policy_Intelligence'
))
DATA_DIR = PROJECT_ROOT / 'data'
sys.path.insert(0, str(PROJECT_ROOT / 'app'))
FILES = {
    '0.6': DATA_DIR / 'ds_vt_ct_csv.CSV',
    '0.8': DATA_DIR / 'ds_vt_ct_0.8_csv.CSV',
    '1.0': DATA_DIR / 'ds_vt_ct_1.0_csv.CSV',
}
assert all(path.exists() for path in FILES.values()), FILES
PROJECT_ROOT

WindowsPath('C:/Users/am.taheri/Documents/AvicenAI/AV_Safety_Policy_Intelligence')

## Data

### 1. Load and normalize the three source tables

In [2]:
frames = {}
for tau, path in FILES.items():
    frame = pd.read_csv(path, sep=';', low_memory=False)
    frame = frame.loc[:, ~frame.columns.astype(str).str.startswith('Unnamed')].copy()
    frame['scenario_number'] = frame['scenario'].round().astype(int)
    frame['tau'] = tau
    frames[tau] = frame

profile = pd.DataFrame([
    {
        'Headway': tau,
        'Rows': len(frame),
        'Columns': frame.shape[1],
        'Within-file exact duplicates': int(frame.duplicated().sum()),
        'Missing cells': int(frame.isna().sum().sum()),
        'Selected events (minTTC < 0.5 s)': int(frame['minTTC'].lt(0.5).sum()),
        'Selected-event share': frame['minTTC'].lt(0.5).mean(),
    }
    for tau, frame in frames.items()
])
profile

,Headway,Rows,Columns,Within-file exact duplicates,Missing cells,Selected events (minTTC < 0.5 s),Selected-event share
0,0.6,97001,18,0,0,25355,0.261389
1,0.8,78648,18,0,0,5587,0.071038
2,1.0,61275,18,0,0,916,0.014949


### 2. Check schema agreement and the repeated S1 baseline

In [3]:
schemas_match = len({tuple(frame.columns) for frame in frames.values()}) == 1
comparison_columns = [column for column in frames['0.6'].columns if column != 'tau']
baseline_06 = frames['0.6'].loc[frames['0.6']['scenario_number'].eq(1), comparison_columns].reset_index(drop=True)
baseline_checks = {
    tau: baseline_06.equals(
        frames[tau].loc[frames[tau]['scenario_number'].eq(1), comparison_columns].reset_index(drop=True)
    )
    for tau in ['0.8', '1.0']
}
raw_merged = pd.concat(frames.values(), ignore_index=True)
repeated_s1 = raw_merged['scenario_number'].eq(1) & raw_merged['tau'].ne('0.6')
modeling_data = raw_merged.loc[~repeated_s1].copy()
{
    'schemas_match_after_empty-column_cleanup': schemas_match,
    'S1_0.8_matches_0.6': baseline_checks['0.8'],
    'S1_1.0_matches_0.6': baseline_checks['1.0'],
    'raw_merged_rows': len(raw_merged),
    'repeated_S1_rows_removed': int(repeated_s1.sum()),
    'final_modeling_rows': len(modeling_data),
    'effective_configurations': modeling_data[['scenario_number', 'tau']].drop_duplicates().shape[0],
}

{'schemas_match_after_empty-column_cleanup': True,
 'S1_0.8_matches_0.6': False,
 'S1_1.0_matches_0.6': False,
 'raw_merged_rows': 236924,
 'repeated_S1_rows_removed': 19314,
 'final_modeling_rows': 217610,
 'effective_configurations': 34}

### 3. Validate target ranges and key category coverage

In [4]:
quality_checks = pd.Series({
    'All minTTC values are within [0, 1]': modeling_data['minTTC'].between(0, 1).all(),
    'All scenarios are within 1-12': modeling_data['scenario_number'].between(1, 12).all(),
    'All headways are recognized': modeling_data['tau'].isin(['0.6', '0.8', '1.0']).all(),
    'No missing values in dashboard source fields': not modeling_data.isna().any().any(),
    'Three ego vehicle classes are present': modeling_data['ego_vtype'].nunique() == 3,
    'Three foe vehicle classes are present': modeling_data['foe_vtype'].nunique() == 3,
})
quality_checks.to_frame('Pass')

,Pass
"All minTTC values are within [0, 1]",True
All scenarios are within 1-12,True
All headways are recognized,True
No missing values in dashboard source fields,True
Three ego vehicle classes are present,True
Three foe vehicle classes are present,True


## Results

### 4. Verify every model mode and headway scope

In [5]:
from ml_modeling import ModelRequest, prepare_model_frame

matrix_checks = []
for mode in ['Microscopic', 'Policy levers', 'Combined']:
    for scope in ['All headways', '0.6 s', '0.8 s', '1.0 s']:
        features, target, groups, metadata = prepare_model_frame(
            raw_merged, ModelRequest('Continuous minTTC', mode, scope)
        )
        matrix_checks.append({
            'Mode': mode,
            'Headway scope': scope,
            'Rows': len(features),
            'Encoded features': features.shape[1],
            'Scenario groups': groups.nunique(),
            'Effective configurations': metadata['effective_configurations'],
        })
pd.DataFrame(matrix_checks)

,Mode,Headway scope,Rows,Encoded features,Scenario groups,Effective configurations
0,Microscopic,All headways,217610,13,12,34
1,Microscopic,0.6 s,97001,13,12,12
2,Microscopic,0.8 s,78648,13,12,12
3,Microscopic,1.0 s,61275,13,12,12
4,Policy levers,All headways,217610,3,12,34
5,Policy levers,0.6 s,97001,2,12,12
6,Policy levers,0.8 s,78648,2,12,12
7,Policy levers,1.0 s,61275,2,12,12
8,Combined,All headways,217610,16,12,34
9,Combined,0.6 s,97001,15,12,12


## Takeaways

- **Completeness:** the required dashboard fields are populated in all three source files.
- **Consistency:** the schemas match after removing three empty trailing columns from the 0.8 file.
- **Duplicate control:** S1 is an exact repeated baseline; removing the 0.8 and 1.0 copies prevents duplicate weighting.
- **Model coverage:** all three feature modes work for the merged dataset and each headway separately.
- **Known gap:** lane count, speed limit, and control type are not available in these dashboard CSVs, so the microscopic mode is reduced and labeled accordingly.
- **Interpretation:** model performance and SHAP outputs are conditional on extracted minTTC ≤ 1.0 s events and must not be read as crash probability or causal effect.